# Outer Skin/Skull Surface Registration — CPD + TPS (simplified, `.k` in / `.vtu` out)
**MacBook M2 / VSCode / Python 3 (arm64)**

Simplified scope of the full pipeline: registers the **outer skull/skin surface
only**. Brain and every other internal part are left at their original template
coordinates, untouched. Element connectivity/topology/PIDs are preserved exactly
— only the node coordinates of the skull-surface node set are updated.

```
Template .k (skull+brain) ──┐
                              ├─ EXTRACT outer skull/skin surface only in VTU of both of it
SAM3D landmarks (subject) ───┘  (brain + internal parts set aside, untouched)
        │
        ▼
Stage 0 — Center + resize
        ▼
Stage 1 — PCA coarse pre-alignment
        ▼
Stage 2 — Rigid CPD
        ▼
Stage 3 — Affine CPD
        ▼
Stage 4 — Non-rigid CPD
        ▼
Stage 5 — TPS propagation (dense outer surface only, NOT the brain)
        ▼
Stage 6 — Non-rigid ICP refinement (lightweight local polish)
        ▼
Stage 7 — Validation (NN / Chamfer / Hausdorff + k_mesh_qa.py)
        ▼
Final .vtu — outer surface node coordinates updated, everything else unchanged
```

**Full-pipeline stages deferred for later** (per your simplification request):
brain/internal-node TPS propagation and the SDF cavity-constraint correction.
Nothing else changes when you add them back — Stage 5 gets extended to the
full mesh and the cavity constraint gets reinserted right after it.


## Setup

On Apple Silicon (M2), `pyvista`/`vtk` and `pycpd` all have native arm64 wheels — no Rosetta/x86 workaround needed. Run this once in your VSCode Python environment (ideally a project-local venv).

In [ ]:
!pip install pycpd pyvista numpy scipy matplotlib pandas -q

import os, sys
import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt
import pandas as pd
from scipy.spatial import cKDTree
from scipy.interpolate import RBFInterpolator
from pycpd import RigidRegistration, AffineRegistration, DeformableRegistration

np.random.seed(0)
print('pyvista:', pv.__version__)
print('numpy  :', np.__version__)


## Configuration

`TEMPLATE_K` is the actual `.k` file we read node coordinates from and write
back to. `TEMPLATE_VTU` is the same mesh pre-converted to VTU (already produced
by `K_to_VTK_Converter.ipynb` in this project) — used only as a geometry
convenience so we can call `extract_surface()` to get the true outer skull
boundary; its `node_id` point-data array is the authoritative link back to the
real `.k` node IDs, so nothing here depends on the two files staying in sync
beyond node IDs/coordinates matching, which they do since the VTU was derived
directly from this `.k`.

`SUBJECT_LANDMARKS_PATH` should point to the SAM3D output once it exists
(`.npy` Nx3, or whitespace/comma-delimited `.csv`/`.xyz`). Until then this
falls back to subsampling an existing head-shape surface in this project as a
placeholder — clearly flagged below — purely so the notebook is runnable and
the whole pipeline can be smoke-tested end to end before SAM3D is wired in.

In [ ]:
BASE_DIR = r'/Volumes/Extreme SSD/2d_3d_regis dataset/New_version'

TEMPLATE_K    = os.path.join(BASE_DIR, 'Head_V6.k')
TEMPLATE_VTU  = os.path.join(BASE_DIR, 'Data/outputs/vtk_export/Head_V6.vtu')

# Outer head-skin surface, cleaned/re-exported as .vtu (fixed 410 degenerate
# quad->triangle elements; see CHANGELOG.md). Still the template's own skin
# surface, not independent SAM3D subject data -- see the "same-coordinate-frame
# sanity check" caveat in CHANGELOG.md -- but this is the file driving
# registration now, per user request.
SUBJECT_LANDMARKS_PATH = r'/Volumes/Extreme SSD/Head_model/head_surface.vtu'
# Placeholder fallback ONLY used if SUBJECT_LANDMARKS_PATH doesn't exist yet:
FALLBACK_SUBJECT_FILE = r'/Volumes/Extreme SSD/Head_model/head_surface.key'  # outer head skin only
N_FALLBACK_SAMPLES = 3000

OUTPUT_DIR = os.path.join(BASE_DIR, 'Data/outputs/cpd_tps_skin_registration')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Outer skull surface PIDs — same convention validated earlier in this project
# (VTU_TwoStage_Registration_v3_3.ipynb): registering the skull's own outer
# surface against the subject's head-shape data, rather than the full facial
# skeleton, keeps the registration signal from being diluted.
SKULL_SURFACE_PID_MIN, SKULL_SURFACE_PID_MAX = 88000001, 88000099

N_LANDMARKS_FIXED  = 2000
N_LANDMARKS_MOVING = 2000

CPD_ALPHA = 2.0
CPD_BETA  = 2.0
CPD_MAX_ITER = 100
CPD_TOL = 1e-6

TPS_SMOOTHING = 1e-3

# Lightweight non-rigid ICP refinement (Stage 6)
ICP_ITERS = 10
# Root-caused via diagnostic: the top-of-head "surplus" region's dense-surface
# points sit ~40-45mm from their true nearest subject point after TPS -- a
# real, large gap, not noise (head_surface.vtu is clean re-exported geometry,
# not a noisy independent scan). The old 15mm cutoff was rejecting this
# correspondence outright as a suspected outlier, permanently locking out the
# correction (confirmed: CPD alpha/beta retuning didn't move the residual at
# all, ruling out CPD's own regularization as the cause -- see CHANGELOG.md).
# Raised so ICP can actually reach real, large, legitimate corrections.
# Revisit downward once real (noisier) SAM3D data replaces this same-frame
# validation dataset -- a 50mm tolerance is only safe because this input has
# no scan noise to accidentally chase.
ICP_MAX_CORR_DIST_MM = 50.0
ICP_STEP = 0.5                # fraction of the pull vector applied per iteration
ICP_SMOOTH_NEIGHBORS = 8      # k-NN graph smoothing to keep the surface coherent

print('Template .k  exists:', os.path.exists(TEMPLATE_K))
print('Template .vtu exists:', os.path.exists(TEMPLATE_VTU))
print('Subject landmarks   :', os.path.exists(SUBJECT_LANDMARKS_PATH), '(fallback will be used if False)')


## Load template geometry + extract head/face surface (jaw + teeth excluded)

**Scope narrowed again, per user request:** the raw PID range
`88000001-88000099` ("facial/cranial bone") includes `mandible` (lower jaw),
`teeth-lwr/upr`, `alveolar`, and `palatine` -- none of these have any real
outer-skin correspondence (the jaw hangs down past where face skin actually
is, teeth are inside the mouth with no skin counterpart at all). Registering
these against a skin surface produced the purple blob hanging down past the
chin into what looked like a spurious "neck" region.

**Fix:** restrict to `HEAD_FACE_INCLUDE_PREFIXES` -- the braincase
(`frontal`, `parietal`, `temporal`, `occipital`, `sphenoid`, `ethmoid`) plus
the mid-face bones that do have a real skin counterpart (`lacrimal`, `nasal`,
`zygomatic`, `maxilla` -- eye socket, nose, cheekbone, upper jaw/cheek area).
Everything else (`mandible`, `palatine`, `teeth-*`, `alveolar`, and the
unlabeled `str*`/`ng-str*` jaw-adjacent parts) is now excluded entirely --
not warped, not driving the registration, frozen at its original position
exactly like the brain.

Same three node sets as before, just over this narrower part list:
- **`skull_surf_*`** -- true external (skin-facing) surface of the
  head+face bones only. Drives the registration.
- **`skull_full_*`** -- every node of the head+face bone volume (external +
  internal shells + solid). Candidate for warping.
- **Frozen boundary nodes** -- any head+face node also touched by brain,
  jaw/teeth, or any other excluded tissue stays at its original position.

In [ ]:
import re

HEAD_FACE_INCLUDE_PREFIXES = (
    'frontal', 'parietal', 'temporal', 'occipital', 'sphenoid', 'ethmoid',  # braincase
    'lacrimal', 'nasal', 'zygomatic', 'maxilla',                            # mid-face, has real skin counterpart
)
# Explicitly excluded: mandible (jaw), palatine, teeth-lwr/upr, alveolar,
# str1-7/ng-str/ng-str2 -- no outer-skin correspondence, caused the
# hanging-down "neck" artifact.

def parse_part_names(k_path):
    names = {}
    with open(k_path, errors='ignore') as f:
        for line in f:
            if line.startswith('$HMNAME COMPS'):
                m = re.match(r'\$HMNAME COMPS\s*(\d+)(.*)', line.strip())
                if m:
                    names[int(m.group(1))] = m.group(2).strip()
    return names

def matches_include_list(name, include_prefixes):
    return any(name.startswith(p) for p in include_prefixes)

PART_NAMES = parse_part_names(TEMPLATE_K)

SKULL_EXTERNAL_PIDS = [
    pid for pid in range(SKULL_SURFACE_PID_MIN, SKULL_SURFACE_PID_MAX + 1)
    if '_external_' in PART_NAMES.get(pid, '') and matches_include_list(PART_NAMES.get(pid, ''), HEAD_FACE_INCLUDE_PREFIXES)
]
HEAD_FACE_ALL_PIDS = [
    pid for pid in range(SKULL_SURFACE_PID_MIN, SKULL_SURFACE_PID_MAX + 1)
    if matches_include_list(PART_NAMES.get(pid, ''), HEAD_FACE_INCLUDE_PREFIXES)
]

template_mesh = pv.read(TEMPLATE_VTU)
print(f'Template: {template_mesh.n_points:,} pts, {template_mesh.n_cells:,} cells')
print(f'External-only head/face PIDs found: {len(SKULL_EXTERNAL_PIDS)} -> {SKULL_EXTERNAL_PIDS}')
print(f'All head/face PIDs (any variant)   : {len(HEAD_FACE_ALL_PIDS)}')

pids = template_mesh.cell_data['part_id']

# External surface only -- drives the registration (real subject correspondence)
ext_cells = np.where(np.isin(pids, SKULL_EXTERNAL_PIDS))[0]
skull_sub = template_mesh.extract_cells(ext_cells)
skull_surf = skull_sub.extract_surface(algorithm='dataset_surface').triangulate()
skull_surf_node_ids = skull_surf.point_data['node_id']
skull_surf_pts0     = np.asarray(skull_surf.points, dtype=np.float64)

# Full head/face bone volume (external + internal shells + solid) -- candidate for warping
full_cells = np.where(np.isin(pids, HEAD_FACE_ALL_PIDS))[0]
skull_full_sub = template_mesh.extract_cells(full_cells)
skull_full_node_ids = np.asarray(skull_full_sub.point_data['node_id'])
skull_full_pts0     = np.asarray(skull_full_sub.points, dtype=np.float64)

full_nid_to_idx = {int(nid): i for i, nid in enumerate(skull_full_node_ids)}
external_idx_in_full = np.array([full_nid_to_idx[int(nid)] for nid in skull_surf_node_ids])

# Freeze any head/face node also referenced by anything OUTSIDE the head/face
# list (brain, jaw, teeth, or anything else) -- warping these tears whichever
# side doesn't move.
sys.path.insert(0, os.path.join(BASE_DIR, 'Code'))
from k_mesh_qa import parse_k_file
_, elems_solid_all, elems_shell_all = parse_k_file(TEMPLATE_K)

head_face_pid_set = set(HEAD_FACE_ALL_PIDS)
head_face_node_set = set(int(n) for n in skull_full_node_ids)
frozen_boundary_nodes = set()
for eid, (pid, conn) in list(elems_solid_all.items()) + list(elems_shell_all.items()):
    if pid not in head_face_pid_set:
        for n in conn:
            if n in head_face_node_set:
                frozen_boundary_nodes.add(n)

warp_mask = np.array([int(nid) not in frozen_boundary_nodes for nid in skull_full_node_ids])

print(f'\nExternal surface (drives registration)      : {skull_surf.n_points:,} pts')
print(f'Full head/face bone volume (candidate)       : {len(skull_full_node_ids):,} pts')
print(f'Frozen at boundary with excluded tissue       : {(~warp_mask).sum():,} pts')
print(f'Actually warped                               : {warp_mask.sum():,} pts')
print(f'Brain + jaw/teeth + everything else: {template_mesh.n_points - len(skull_full_node_ids):,} pts left untouched')

## Load subject landmarks (SAM3D, with placeholder fallback)

In [ ]:
def load_subject_landmarks(path, fallback_path, n_fallback):
    def _load_mesh_points(p):
        if p.endswith(('.k', '.key')):
            sys.path.insert(0, os.path.join(BASE_DIR, 'Code'))
            from k_mesh_qa import parse_k_file
            pts_dict, _, _ = parse_k_file(p)
            return np.array(list(pts_dict.values()), dtype=np.float64)
        mesh = pv.read(p)
        return np.asarray(mesh.points, dtype=np.float64)

    if os.path.exists(path):
        if path.endswith('.npy'):
            pts = np.load(path)
        elif path.endswith(('.csv', '.xyz', '.txt')):
            pts = np.loadtxt(path, delimiter=',') if path.endswith('.csv') else np.loadtxt(path)
        elif path.endswith(('.vtu', '.vtk', '.stl', '.ply', '.obj', '.k', '.key')):
            # Mesh formats -- use ALL points at full density (no subsampling
            # here; Stage 0.5's farthest-point sampling already handles the
            # landmark subset needed for CPD, and the dense ICP/blend/
            # validation stages want the full surface, not a random subsample).
            pts = _load_mesh_points(path)
        else:
            raise ValueError(f'Unrecognized subject landmarks format: {path}')
        print(f'Loaded {len(pts):,} subject landmarks from {path}')
        return np.asarray(pts, dtype=np.float64)

    print(f'⚠ {path} not found — SAM3D landmarks not wired in yet.')
    print(f'  Falling back to {fallback_path}')
    print('  Replace SUBJECT_LANDMARKS_PATH once SAM3D output is available.')

    fb_pts = _load_mesh_points(fallback_path)
    print(f'  Fallback surface: {len(fb_pts):,} points total')
    idx = np.random.choice(len(fb_pts), min(n_fallback, len(fb_pts)), replace=False)
    return fb_pts[idx]

subject_pts = load_subject_landmarks(SUBJECT_LANDMARKS_PATH, FALLBACK_SUBJECT_FILE, N_FALLBACK_SAMPLES)
print(f'Subject point set: {subject_pts.shape}')

## Stage 0 — Center + resize

Center both point sets at their own centroid; scale the template skull surface
so its bounding-box diagonal matches the subject's.

In [ ]:
def center_and_scale(fixed_pts, moving_pts):
    c_fixed  = fixed_pts.mean(0)
    c_moving = moving_pts.mean(0)
    f_c = fixed_pts - c_fixed
    m_c = moving_pts - c_moving
    diag_f = np.linalg.norm(f_c.max(0) - f_c.min(0))
    diag_m = np.linalg.norm(m_c.max(0) - m_c.min(0))
    scale = diag_f / (diag_m + 1e-8)
    return f_c, m_c * scale, dict(c_fixed=c_fixed, c_moving=c_moving, scale=scale)

# Basis for centering/scale is still the EXTERNAL surface vs subject (the only
# pair with real correspondence) -- the full skull volume is then carried
# through the exact same transform so everything stays in one consistent frame.
nf_c, nm_surf_cs, norm_params = center_and_scale(subject_pts, skull_surf_pts0)
c_fixed, c_moving, scale = norm_params['c_fixed'], norm_params['c_moving'], norm_params['scale']
nm_full_cs = (skull_full_pts0 - c_moving) * scale

print(f'Centroid (subject) : {c_fixed.round(1)}')
print(f'Centroid (template): {c_moving.round(1)}')
print(f'Scale factor        : {scale:.4f}')

def plot_overlap(a, b, title, labels=('Subject', 'Template'), n_sample=4000):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    views = [(0, 1, 'Axial X-Y'), (0, 2, 'Coronal X-Z'), (1, 2, 'Sagittal Y-Z')]
    sa = np.random.choice(len(a), min(n_sample, len(a)), replace=False)
    sb = np.random.choice(len(b), min(n_sample, len(b)), replace=False)
    for ax, (i, j, lbl) in zip(axes, views):
        ax.scatter(a[sa, i], a[sa, j], s=1, c='red', alpha=0.3, label=labels[0])
        ax.scatter(b[sb, i], b[sb, j], s=1, c='steelblue', alpha=0.3, label=labels[1])
        ax.set_title(lbl); ax.set_aspect('equal'); ax.legend(markerscale=8)
    fig.suptitle(title)
    plt.tight_layout(); plt.show()

plot_overlap(nf_c, nm_surf_cs, 'After center + resize (before any registration)')

## Stage 0.5 — Landmark subsampling (farthest-point sampling)

CPD cost is O(N·M) per iteration, so registration runs on a bounded landmark
subset; TPS (Stage 5) propagates the result to the full dense surface.

In [ ]:
def farthest_point_sampling(points, n_samples, seed=0):
    rng = np.random.default_rng(seed)
    n = len(points)
    n_samples = min(n_samples, n)
    selected = np.empty(n_samples, dtype=np.int64)
    selected[0] = rng.integers(n)
    dist = np.linalg.norm(points - points[selected[0]], axis=1)
    for k in range(1, n_samples):
        selected[k] = np.argmax(dist)
        dist = np.minimum(dist, np.linalg.norm(points - points[selected[k]], axis=1))
    return selected

fixed_lm_idx  = farthest_point_sampling(nf_c, N_LANDMARKS_FIXED)
moving_lm_idx = farthest_point_sampling(nm_surf_cs, N_LANDMARKS_MOVING)

fixed_lm  = nf_c[fixed_lm_idx]
moving_lm = nm_surf_cs[moving_lm_idx]

print(f'Fixed  (subject) landmarks : {len(fixed_lm):,}')
print(f'Moving (template) landmarks: {len(moving_lm):,}')
plot_overlap(fixed_lm, moving_lm, 'Landmark subsets (FPS)')


## Stage 1 — PCA coarse pre-alignment

CPD is a local optimizer and heads are near-bilaterally-symmetric — without a
decent initial pose, rigid CPD can converge to a self-consistent but wrong
rotation. PCA-align the principal axes first, resolving the sign/reflection
ambiguity by testing the 4 valid proper-rotation candidates against a quick
nearest-neighbor check.

In [ ]:
def pca_axes(pts):
    cov = np.cov(pts.T)
    w, v = np.linalg.eigh(cov)
    order = np.argsort(w)[::-1]
    return v[:, order]

def pca_prealign(fixed_pts, moving_pts, eval_stride=5):
    V_f = pca_axes(fixed_pts)
    V_m = pca_axes(moving_pts)
    tree = cKDTree(fixed_pts)
    best = None
    for s0 in (1, -1):
        for s1 in (1, -1):
            for s2 in (1, -1):
                S = np.diag([s0, s1, s2])
                R = V_f @ S @ V_m.T
                if np.linalg.det(R) < 0:
                    continue  # reflection, not a valid rotation
                cand = moving_pts @ R.T
                d, _ = tree.query(cand[::eval_stride])
                score = d.mean()
                if best is None or score < best[0]:
                    best = (score, R)
    return best[1], best[0]

R_pca, pca_score = pca_prealign(fixed_lm, moving_lm)
print(f'PCA pre-alignment rotation (best of 4 valid sign combinations):')
print(np.round(R_pca, 3))
print(f'Mean NN distance after PCA align: {pca_score:.2f} (normalized units)')

moving_lm_pca  = moving_lm  @ R_pca.T
nm_surf_pca    = nm_surf_cs @ R_pca.T   # dense external surface, consistent with the landmark subset
nm_full_pca    = nm_full_cs @ R_pca.T   # full skull bone volume (external+internal+solid), same transform

plot_overlap(fixed_lm, moving_lm_pca, 'After PCA coarse pre-alignment')


## Stage 2 — Rigid CPD

In [ ]:
rigid_reg = RigidRegistration(X=fixed_lm, Y=moving_lm_pca, max_iterations=CPD_MAX_ITER, tolerance=CPD_TOL)
TY_rigid, (s_r, R_r, t_r) = rigid_reg.register()

print(f'Rigid scale       : {s_r:.4f}')
print(f'Rigid rotation R  :\n{np.round(R_r, 3)}')
print(f'Rigid translation : {np.round(t_r, 2)}')

plot_overlap(fixed_lm, TY_rigid, 'After rigid CPD registration')


## Stage 3 — Affine CPD

In [ ]:
AFFINE_SINGULAR_VALUE_CLAMP = (0.8, 1.25)  # reject implausible shear/scale (overfitting a sparse landmark set)

affine_reg = AffineRegistration(X=fixed_lm, Y=TY_rigid, max_iterations=CPD_MAX_ITER, tolerance=CPD_TOL)
_, (B_a, t_a) = affine_reg.register()

# Clamp singular values of B: an unconstrained affine fit on a small/sparse
# landmark set can overfit into an anatomically implausible shear or
# axis-scale (observed: up to 41% z-scale + heavy shear on this dataset,
# producing a >150mm Hausdorff blow-up downstream). Clamping keeps the
# transform close to similarity while still allowing modest anisotropy.
U, S, Vt = np.linalg.svd(B_a)
S_clamped = np.clip(S, *AFFINE_SINGULAR_VALUE_CLAMP)
B_a_clamped = U @ np.diag(S_clamped) @ Vt
TY_affine_candidate = TY_rigid @ B_a_clamped + t_a

print(f'Affine matrix B (raw)    :\n{np.round(B_a, 3)}')
print(f'Singular values (raw)    : {np.round(S, 3)}')
print(f'Singular values (clamped): {np.round(S_clamped, 3)}')
print(f'Affine translation: {np.round(t_a, 2)}')

# Regression guard: even clamped, affine CPD can still fit the landmark
# subset worse than rigid CPD alone did (observed on this dataset: Hausdorff
# regressed 52.2mm -> 75.7mm despite clamping). The clamp bounds *how
# implausible* the transform can be, but doesn't guarantee it improves the
# fit -- so directly compare against rigid CPD's own landmark residual and
# skip affine entirely (fall back to TY_rigid) if it's worse. This is what
# was silently propagating a corrupted global alignment all the way through
# deformable CPD/TPS into the confidence blend, and is the actual root cause
# of the "surplus above the head" (a coherent misaligned region, not a
# single-point tear -- confirmed by Stage 6.4's local-outlier cap firing on
# zero points against this same bad alignment).
def landmark_hausdorff(a, b):
    tree = cKDTree(b)
    d, _ = tree.query(a)
    return d.max()

haus_rigid  = landmark_hausdorff(TY_rigid, fixed_lm)
haus_affine = landmark_hausdorff(TY_affine_candidate, fixed_lm)
affine_regressed = haus_affine > haus_rigid
TY_affine = TY_rigid if affine_regressed else TY_affine_candidate

print(f'\nLandmark Hausdorff -- rigid: {haus_rigid:.2f}, affine (clamped): {haus_affine:.2f}')
print(f'Affine regressed vs rigid: {affine_regressed} -> {"falling back to rigid" if affine_regressed else "keeping affine"}')

plot_overlap(fixed_lm, TY_affine, 'After affine CPD registration (clamped, regression-guarded)')

## Stage 4 — Non-rigid (deformable) CPD

In [ ]:
cpd_errors = []
def _track(iteration, error, X, Y):
    cpd_errors.append(error)

deformable_reg = DeformableRegistration(X=fixed_lm, Y=TY_affine,
                                         alpha=CPD_ALPHA, beta=CPD_BETA,
                                         max_iterations=CPD_MAX_ITER, tolerance=CPD_TOL)
TY_cpd, (G, W) = deformable_reg.register(callback=_track)

plt.figure(figsize=(6, 3))
plt.plot(cpd_errors)
plt.xlabel('EM iteration'); plt.ylabel('CPD objective (negative log-likelihood)')
plt.title('Deformable CPD convergence'); plt.tight_layout(); plt.show()

plot_overlap(fixed_lm, TY_cpd, 'After deformable CPD registration')


## Stage 5 — TPS propagation (outer surface only, brain untouched)

Fit TPS on the Stage-4 landmark displacements, then evaluate it on the **full
dense outer skull surface** only — `nm_surf_pca` (all ~20k surface points),
not the brain and not any other internal structure. Anything not part of
`skull_surf_node_ids` is never touched by this notebook.

In [ ]:
tps = RBFInterpolator(moving_lm, TY_cpd, kernel='thin_plate_spline', smoothing=TPS_SMOOTHING)

# Evaluate on the FULL skull bone volume (external + internal shells + solid),
# not just the external surface -- this is what keeps the bone's own solid
# elements from stretching between a moved external face and a fixed internal
# face (root cause of ~2,000 newly-inverted elements, see changelog).
skull_full_warped_cs = tps(nm_full_pca)
nm_surf_warped_cs = skull_full_warped_cs[external_idx_in_full]  # external subset, for Stage 6/6.5 local refinement

print(f'Warped full skull volume : {skull_full_warped_cs.shape[0]:,} nodes')
print(f'Warped external surface  : {nm_surf_warped_cs.shape[0]:,} nodes (subset used for local refinement)')
plot_overlap(nf_c, nm_surf_warped_cs, 'After TPS propagation (external surface vs subject)')

## Stage 6 — Non-rigid ICP refinement (lightweight)

Local polish: pull each warped surface point toward its nearest subject point,
ignoring correspondences beyond `ICP_MAX_CORR_DIST_MM` (avoids chasing noise/
outliers), then k-NN graph-smooth the pull field so the surface moves
coherently rather than point-by-point. Applied over a few damped iterations.

This is a simplified stand-in for the full Amberg et al. (2007) optimal-step
non-rigid ICP — good enough for a local polish pass, but if you need the
published stiffness-relaxation formulation later, that's the upgrade path.

In [ ]:
def graph_smooth(vectors, points, k=8, iters=1):
    tree = cKDTree(points)
    _, idx = tree.query(points, k=k + 1)  # includes self
    smoothed = vectors.copy()
    for _ in range(iters):
        smoothed = smoothed[idx].mean(axis=1)
    return smoothed

def nonrigid_icp_refine(source_pts, target_pts, iters, max_corr_dist_mm, step, scale, k_smooth):
    pts = source_pts.copy()
    tree_t = cKDTree(target_pts)
    for it in range(iters):
        d, nn_idx = tree_t.query(pts)
        pull = target_pts[nn_idx] - pts
        valid = (d / scale) < max_corr_dist_mm   # scale converts normalized units back to mm
        pull[~valid] = 0.0
        pull_smooth = graph_smooth(pull, pts, k=k_smooth)
        pts = pts + step * pull_smooth
    return pts

nm_surf_icp_cs = nonrigid_icp_refine(
    nm_surf_warped_cs, nf_c,
    iters=ICP_ITERS, max_corr_dist_mm=ICP_MAX_CORR_DIST_MM,
    step=ICP_STEP, scale=scale, k_smooth=ICP_SMOOTH_NEIGHBORS,
)

plot_overlap(nf_c, nm_surf_icp_cs, 'After non-rigid ICP refinement')


## Stage 6.4 — Displacement-magnitude cap (physical-plausibility safeguard)

**Why this is needed, on top of Stage 6.5:** the confidence-damped blend below
only compares *warped-vs-original distance to the subject target*. It has no
way to know whether the target correspondence itself is trustworthy. We are
currently running against `head_surface.key` — the template's *own* skin
surface, re-exported (see `CHANGELOG.md`); real SAM3D subject landmarks
(`Data/subject_landmarks.npy`) don't exist yet. That means there is weak/no
genuine correspondence signal at the top of the skull (vertex region), so
CPD/TPS can extrapolate wildly there — and because the wrong warped point can
still land *closer* to some nearby wrong target point than the original does,
Stage 6.5's relative "which is better" comparison alone doesn't catch it. This
is the purple spike/surplus poking above the skin seen in the rendered mesh.

**First attempt (wrong, kept here for the record): a fixed absolute-mm cap.**
Tried capping every point's displacement at a flat `MAX_DISPLACEMENT_MM`
(e.g. 40mm). This clamped 9,885 of 10,029 points (98.6%) — because the
legitimate *global* pose correction from Stages 2-4 alone already moves
almost every point more than 40mm. An absolute physical threshold can't tell
"this whole surface needed a big rigid+affine correction" apart from "this
one point flew off on its own" — it penalizes both identically. Inverted
elements barely changed (7,764 → 7,828).

**Actual fix: a local-outlier-relative cap.** What tears elements isn't a
point moving far in absolute terms — it's a point moving *inconsistently
with its mesh neighbors* (exactly what Stage 6.5's confidence-smoothing
already targets for the blend decision, but not for the underlying warped
position itself). So instead: for each point, compute the mean displacement
of its `k`-nearest neighbors, and cap that point's own displacement magnitude
at `LOCAL_OUTLIER_RATIO ×` its neighborhood's mean (with a small absolute
`LOCAL_OUTLIER_FLOOR_MM` so tightly-clustered small variations aren't falsely
flagged). A point that's moving in step with its neighbors — even by 60-80mm,
as part of a real global correction — passes through untouched. A point that
diverges sharply from its neighbors (the spike) gets pulled back toward
neighborhood-consistent magnitude. Applied before Stage 6.5, so the blend's
own "which is closer" logic then runs on an already-coherent displacement
field.</cell id="97cea079">

In [ ]:
LOCAL_OUTLIER_RATIO = 1.6     # a point may not move more than this multiple of its k-NN neighborhood's mean displacement
LOCAL_OUTLIER_FLOOR_MM = 10.0 # absolute floor so tightly-clustered small displacements aren't falsely flagged
LOCAL_OUTLIER_K = 8

def clamp_local_outlier_displacement(original_pts, warped_pts, scale, k=LOCAL_OUTLIER_K,
                                      ratio=LOCAL_OUTLIER_RATIO, floor_mm=LOCAL_OUTLIER_FLOOR_MM):
    disp = warped_pts - original_pts
    tree = cKDTree(original_pts)
    _, idx = tree.query(original_pts, k=k + 1)   # includes self in column 0
    neighbor_disp_mean = disp[idx[:, 1:]].mean(axis=1)

    point_mag_mm    = np.linalg.norm(disp, axis=1) / scale
    neighbor_mag_mm = np.linalg.norm(neighbor_disp_mean, axis=1) / scale
    threshold_mm = np.maximum(neighbor_mag_mm * ratio, floor_mm)

    over = point_mag_mm > threshold_mm
    disp = disp.copy()
    if over.any():
        shrink = (threshold_mm[over] / point_mag_mm[over])[:, None]  # scale-agnostic, <1 by construction
        disp[over] = disp[over] * shrink
    return original_pts + disp, over

nm_surf_icp_cs, local_outlier_capped = clamp_local_outlier_displacement(nm_surf_pca, nm_surf_icp_cs, scale)

print(f'Local-outlier cap applied to {local_outlier_capped.sum():,} / {len(local_outlier_capped):,} points '
      f'(ratio={LOCAL_OUTLIER_RATIO}, floor={LOCAL_OUTLIER_FLOOR_MM}mm, k={LOCAL_OUTLIER_K})')
plot_overlap(nf_c, nm_surf_icp_cs, 'After local-outlier displacement cap')

## Stage 6.5 -- Confidence-damped blend (do-no-harm safeguard)

Any surface region with no nearby subject correspondence (missing coverage in
the subject scan, or a genuinely hard local fit e.g. near the ear/mastoid/neck
base) can get a large, sometimes wild residual displacement out of CPD/TPS/ICP.

**Two separate bugs found and fixed here, in order:**

1. The first version reverted to the *original* position whenever the warp's
   residual exceeded a fixed threshold, without checking whether the original
   was even farther away (it often was). Fixed by comparing warped vs.
   original residual directly and keeping whichever is closer.
2. **This is the one causing the spikes/self-intersecting geometry visible in
   the rendered result** (a thin rod poking through the ear, a jagged purple
   spike at the top of the skull) -- not a mesh-density problem. The
   per-point confidence was computed *independently* at every point, with no
   regard for its mesh neighbors. Two vertices connected by the same element
   could end up with very different confidence (one kept its full warp, its
   neighbor reverted to original) -- stretching or inverting the element
   between them into exactly this kind of spike. Fixed by spatially smoothing
   the confidence field across the mesh (same `graph_smooth` k-NN smoothing
   already used for the ICP stage) before applying the blend, so neighboring
   points transition gradually instead of jumping.

In [ ]:
BLEND_TRANSITION_MM = 10.0   # width of the blend zone around the "which is better" boundary
CONF_SMOOTH_NEIGHBORS = 8    # spatial smoothing of the confidence field (prevents spikes between neighbors)
CONF_SMOOTH_ITERS = 3

def coverage_damped_blend(original_pts, warped_pts, target_pts, scale,
                           transition_mm=BLEND_TRANSITION_MM,
                           smooth_k=CONF_SMOOTH_NEIGHBORS, smooth_iters=CONF_SMOOTH_ITERS):
    tree_t = cKDTree(target_pts)
    d_orig_mm, _   = tree_t.query(original_pts)
    d_warped_mm, _ = tree_t.query(warped_pts)
    d_orig_mm   /= scale
    d_warped_mm /= scale

    # raw, per-point confidence in the WARP: 1 if warp is clearly better than
    # original, 0 if original is clearly better, linear blend in between
    advantage = d_orig_mm - d_warped_mm   # positive => warp is better
    confidence_raw = np.clip(advantage / transition_mm + 0.5, 0.0, 1.0)

    # Spatially smooth the confidence field itself (k-NN graph, on the
    # WARPED point positions -- these are what the final mesh connectivity
    # actually sits on). Without this, two mesh-adjacent vertices can get
    # very different confidence and the element between them stretches into
    # a spike/self-intersection when one point keeps its warp and its
    # neighbor reverts to original.
    tree_p = cKDTree(warped_pts)
    _, knn_idx = tree_p.query(warped_pts, k=smooth_k + 1)  # includes self
    confidence = confidence_raw.copy()
    for _ in range(smooth_iters):
        confidence = confidence[knn_idx].mean(axis=1)

    blended = original_pts + confidence[:, None] * (warped_pts - original_pts)

    d_blended_mm, _ = tree_t.query(blended)
    d_blended_mm /= scale
    return blended, confidence, confidence_raw, d_orig_mm, d_warped_mm, d_blended_mm

nm_surf_final_cs, coverage_conf, coverage_conf_raw, d_orig_mm, d_warped_mm, d_blended_mm = coverage_damped_blend(
    nm_surf_pca, nm_surf_icp_cs, nf_c, scale)

n_warp_kept = int((coverage_conf >= 0.999).sum())
n_reverted  = int((coverage_conf <= 0.001).sum())
n_blend     = len(coverage_conf) - n_warp_kept - n_reverted
print(f'Warp kept in full (warp was better)  : {n_warp_kept:,} / {len(coverage_conf):,} points')
print(f'Reverted to original (original was better): {n_reverted:,} / {len(coverage_conf):,} points')
print(f'Partially blended                     : {n_blend:,} / {len(coverage_conf):,} points')

# how much did smoothing actually change vs the raw per-point decision?
conf_delta = np.abs(coverage_conf - coverage_conf_raw)
print(f'Confidence changed by smoothing -- mean {conf_delta.mean():.3f}, max {conf_delta.max():.3f}, '
      f'{(conf_delta > 0.1).sum():,} points shifted by >0.1')

print(f'Do-no-harm check -- mean residual: original={d_orig_mm.mean():.2f}mm  warped={d_warped_mm.mean():.2f}mm  blended={d_blended_mm.mean():.2f}mm')
print(f'                     max residual : original={d_orig_mm.max():.2f}mm  warped={d_warped_mm.max():.2f}mm  blended={d_blended_mm.max():.2f}mm')
# NOTE: smoothing the confidence field trades a small amount of the strict
# per-point do-no-harm guarantee for spatial coherence (no more spikes) --
# a smoothed point can end up slightly worse than min(orig, warped) at that
# single point, but neighboring elements no longer tear apart from each other.

plot_overlap(nf_c, nm_surf_final_cs, 'After confidence-damped blend (final, spatially smoothed)')

## Stage 7 — Validation & summary

In [ ]:
def surface_agreement(source_pts, target_pts, scale):
    tree_t = cKDTree(target_pts)
    tree_s = cKDTree(source_pts)
    d_s2t, _ = tree_t.query(source_pts)
    d_t2s, _ = tree_s.query(target_pts)
    d_s2t /= scale; d_t2s /= scale
    chamfer = d_s2t.mean() + d_t2s.mean()
    hausdorff = max(d_s2t.max(), d_t2s.max())
    return dict(mean=d_s2t.mean(), median=np.median(d_s2t), rms=np.sqrt((d_s2t**2).mean()),
                max=d_s2t.max(), chamfer=chamfer, hausdorff=hausdorff)

rows = []
for stage, pts in [
    ('0. Center+resize',        moving_lm),
    ('1. PCA pre-align',        moving_lm_pca),
    ('2. Rigid CPD',            TY_rigid),
    ('3. Affine CPD',           TY_affine),
    ('4. Non-rigid CPD',        TY_cpd),
    ('5. TPS (dense surface)',  nm_surf_warped_cs),
    ('6. Non-rigid ICP',         nm_surf_icp_cs),
    ('6.5 Confidence blend (final)', nm_surf_final_cs),
]:
    rows.append(dict(stage=stage, **surface_agreement(pts, nf_c, scale)))

summary_df = pd.DataFrame(rows).round(3)
summary_df


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(summary_df['stage'], summary_df['mean'], color='steelblue', alpha=0.85)
ax.set_ylabel('Mean nearest-neighbor distance to subject surface (mm)')
ax.set_title('Registration quality by stage (lower = better)')
plt.xticks(rotation=25, ha='right')
plt.tight_layout(); plt.show()

improvement = (summary_df.iloc[0]['mean'] - summary_df.iloc[-1]['mean']) / summary_df.iloc[0]['mean'] * 100
print(f"Mean surface error: {summary_df.iloc[0]['mean']:.2f} mm -> {summary_df.iloc[-1]['mean']:.2f} mm "
      f"({improvement:.1f}% reduction)")
print(f"Final Hausdorff distance: {summary_df.iloc[-1]['hausdorff']:.2f} mm")


## Save outputs -- write a `.vtu`

Combine the two node sets back together: **external surface** nodes get the
fully refined position (TPS + ICP + confidence blend, i.e. `nm_surf_final_cs`
scattered back via `external_idx_in_full`); every other skull node (internal
shell + solid bone volume) gets the Stage-5 TPS-only position
(`skull_full_warped_cs`) -- smooth and coherent with its external neighbor,
since both originate from the same TPS field, so the bone volume elements
stay valid. Undo center/scale/PCA to return to world coordinates. Brain and
everything outside the skull PID range is still completely untouched.

**Tried and reverted: a per-node bone-thickness stretch cap.** Attempted to
fix the inversion increase that came with widening `ICP_MAX_CORR_DIST_MM` by
capping each node's distance from its nearest frozen neighbor (sharing a
solid/shell element) at a multiple of their original, undeformed distance.
Made things worse (7,788 → 9,116 inversions) with no accuracy benefit
(Hausdorff unchanged, since the cap never touches the external surface being
validated). Root cause of the failure: 95.5% of the mixed frozen/moving
elements have *multiple* distinct frozen nodes, not one — capping each moving
node independently toward its own single nearest frozen anchor let different
nodes on the same element snap toward different anchors, turning a smooth,
coherent TPS stretch into a skewed, twisted one. See `CHANGELOG.md` for the
full write-up and what a correct version would need to do differently
(treat the whole frozen face as one anchor, not per-node).</cell id="73375c74">

In [ ]:
def to_world(pts_cs_pca):
    pts_cs = pts_cs_pca @ np.linalg.inv(R_pca).T   # undo PCA rotation
    return (pts_cs / scale) + c_moving             # undo scale + centering

# Start from the TPS-only warp for the whole skull volume, then overwrite the
# external-surface rows with the fully refined (ICP + confidence-blend) result
skull_full_final_cs = skull_full_warped_cs.copy()
skull_full_final_cs[external_idx_in_full] = nm_surf_final_cs

final_world_coords = to_world(skull_full_final_cs)

# Only nodes NOT frozen at the skull/brain interface actually get updated;
# frozen boundary nodes keep their original .k coordinate untouched.
updated_coords = {
    int(nid): xyz for nid, xyz, keep in zip(skull_full_node_ids, final_world_coords, warp_mask) if keep
}
print(f'Nodes to update (full skull bone volume, excluding frozen boundary): {len(updated_coords):,}')

# Map node_id -> point index in the full template mesh, then patch just those rows
node_id_to_idx = {int(nid): idx for idx, nid in enumerate(template_mesh.point_data['node_id'])}

final_mesh = template_mesh.copy()
new_points = final_mesh.points.copy()
for nid, xyz in updated_coords.items():
    new_points[node_id_to_idx[nid]] = xyz
final_mesh.points = new_points

OUTPUT_VTU = os.path.join(OUTPUT_DIR, 'Head_V6_skin_registered.vtu')
final_mesh.save(OUTPUT_VTU)
print(f'Wrote -> {OUTPUT_VTU}')

out_csv = os.path.join(OUTPUT_DIR, 'validation_summary.csv')
summary_df.to_csv(out_csv, index=False)
print(f'Wrote -> {out_csv}')

## QA the written mesh (k_mesh_qa.py)

Sanity-check that patching only the skull-surface node coordinates didn't
invert or degenerate any elements. Reuses the *original* `.k`'s element
connectivity (topology is unchanged) with the *updated* node coordinates —
no need to re-read the `.vtu` back into k_mesh_qa's format.

In [ ]:
sys.path.insert(0, os.path.join(BASE_DIR, 'Code'))
from k_mesh_qa import parse_k_file, check_mesh

nodes_orig, elems_solid_orig, elems_shell_orig = parse_k_file(TEMPLATE_K)

# Same nodes as original, except the skull-surface subset we just warped
nodes_new = dict(nodes_orig)
nodes_new.update(updated_coords)

print(f'Node count  -> original: {len(nodes_orig):,}, updated: {len(nodes_new):,}')
print(f'Solid elems : {len(elems_solid_orig):,}   Shell elems: {len(elems_shell_orig):,}  (topology unchanged)')

result = check_mesh(nodes_new, elems_solid_orig, label=OUTPUT_VTU, elems_shell=elems_shell_orig)

# Confirm brain/internal nodes are truly untouched (not in updated_coords)
untouched_ok = all(nodes_orig[nid] == nodes_new[nid] for nid in nodes_orig if nid not in updated_coords)
print(f'\nAll non-skull-surface nodes unchanged: {untouched_ok}')